# 04 — CNN Baseline

Deep-learning baseline for the GTZAN music genre classification project.

**Run:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all.


In [ ]:
!pip -q install librosa soundfile scikit-learn pandas matplotlib
import os, sys
from pathlib import Path

REPO = Path('/content/music-genre-classification')
if not REPO.exists():
    !git clone -b feature/deep-learning https://github.com/SaimaKapoor/music-genre-classification.git /content/music-genre-classification
sys.path.insert(0, str(REPO))
print('Repository:', REPO)


## 1. Download the raw GTZAN audio
The audio stays in the temporary Colab runtime and is not uploaded to GitHub.


In [ ]:
RAW_URL = 'https://ibm.ent.box.com/index.php?rm=box_download_shared_file&shared_name=gvkgmb4n0h6rbeccdwtwa0ujjjfmouof&file_id=f_961929491012'
ARCHIVE = '/content/genres.tar.gz'
if not os.path.exists(ARCHIVE):
    !wget -O /content/genres.tar.gz "$RAW_URL"
!mkdir -p /content/gtzan
!tar -xzf /content/genres.tar.gz -C /content/gtzan
!find /content/gtzan -maxdepth 2 -type d | head -20


## 2. Upload the shared `split_manifest.csv`
Choose the CSV you generated earlier.


In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'split_manifest.csv' in uploaded, 'Please select split_manifest.csv'
MANIFEST = '/content/split_manifest.csv'


In [ ]:
import pandas as pd
manifest = pd.read_csv(MANIFEST)
print('Total songs:', len(manifest))
print('\nSplit counts:')
print(manifest['split'].value_counts())
print('\nGenre × split:')
print(pd.crosstab(manifest['genre'], manifest['split']))
assert len(manifest) == 1000
assert manifest['relative_path'].is_unique
assert set(manifest['split']) == {'train', 'validation', 'test'}


## 3. Create PyTorch datasets and loaders


In [ ]:
import torch
from torch.utils.data import DataLoader
from src.deep_learning.dataset import GTZANMelDataset
from src.deep_learning.cnn import BaselineCNN
from src.deep_learning.train import fit
from src.deep_learning.evaluate import predict, classification_metrics

train_manifest = manifest[manifest['split'] == 'train'].copy()
val_manifest = manifest[manifest['split'] == 'validation'].copy()
test_manifest = manifest[manifest['split'] == 'test'].copy()

train_ds = GTZANMelDataset(train_manifest, '/content/gtzan', random_windows=True)
val_ds = GTZANMelDataset(val_manifest, '/content/gtzan', random_windows=False)
test_ds = GTZANMelDataset(test_manifest, '/content/gtzan', random_windows=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print('Train songs:', len(train_ds))
print('Validation songs:', len(val_ds))
print('Test songs:', len(test_ds))


## 4. Inspect one log-mel spectrogram


In [ ]:
import matplotlib.pyplot as plt

x, y = train_ds[0]
print('Input tensor shape:', tuple(x.shape))
print('Genre index:', int(y))

plt.figure(figsize=(10, 4))
plt.imshow(x.squeeze(0).numpy(), aspect='auto', origin='lower')
plt.title('Normalized log-mel spectrogram')
plt.xlabel('Time frames')
plt.ylabel('Mel bins')
plt.tight_layout()
plt.show()


## 5. Train the baseline CNN

Baseline settings: 3 convolution blocks, max pooling, ReLU, dropout=0.30, Adam, learning rate=1e-3, batch size=32, 20 epochs.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

model = BaselineCNN(num_classes=10, dropout=0.30)
history = fit(
    model,
    train_loader,
    val_loader,
    epochs=20,
    learning_rate=1e-3,
    device=device,
    checkpoint_path='/content/best_cnn_baseline.pt'
)


## 6. Evaluate the held-out test songs


In [ ]:
y_true, y_pred = predict(model, test_loader, device=device)
metrics = classification_metrics(y_true, y_pred)
print(f"Test accuracy: {metrics['accuracy']:.4f}")
print(f"Test macro-F1: {metrics['macro_f1']:.4f}")
print('\nClassification report:')
print(metrics['report'])


In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv('/content/cnn_baseline_history.csv', index=False)
print('Saved /content/cnn_baseline_history.csv')
